# Training on a public dataset: bev-decision-150K

[avbiswas/bev-decision-150K](https://huggingface.co/datasets/avbiswas/bev-decision-150K) holds
about 150,000 texts from 18 domains (spatial reasoning, sentiment, retail, support routing,
finance, ...), each with one or more typed questions and their correct answers. It is much harder
than the synthetic tickets in `quickstart.ipynb`: the questions differ from row to row, and many
need real reading comprehension.

This notebook trains Qwen3-0.6B with `tuning="head"` (frozen backbone, only the decision head
learns) and compares it with random guessing on held-out test rows, per question type and per
domain. The defaults take about 10–15 minutes on a Mac.

In [ ]:
import json

from datasets import load_dataset

from sysone import DecoderModel, baselines, compare

N_TRAIN = 3000   # training rows (about 2 questions each); the full train split has 125,614
N_TEST = 1000    # held-out rows from the test split

## Load the data

The correct answer is stored as a `label` inside each question, which sysone reads directly. Each
row's domain is passed as its `category`, so validation is split evenly across domains and
results can be broken down by domain.

In [ ]:
def load_rows(split, n):
    ds = load_dataset("avbiswas/bev-decision-150K", split=split).shuffle(seed=0).select(range(n))
    return [{"state": r["state"], "questions": json.loads(r["questions_json"]), "category": r["domain"]}
            for r in ds]


train = load_rows("train", N_TRAIN)
test = load_rows("test", N_TEST)
print("%d training rows, %d test rows" % (len(train), len(test)))
next(iter(train[0]["questions"].values()))

## Before training: random guessing

Questions differ from row to row here, so the only meaningful baseline is random guessing: a
question with k options is right 1/k of the time.

In [ ]:
results = baselines(test)
compare(results)

## Train

The backbone reads every training question once and its outputs are cached; each epoch then
trains only the head. 10% of rows are held back for validation and early stopping.

In [ ]:
model = DecoderModel(tuning="head")
model.fit(train)

## After training

In [ ]:
results["trained"] = model.evaluate(test)
compare(results)

**Metrics** (from `model.evaluate`, the same for hard and soft labels):
- **accuracy**: the model's most likely option is the right one (for soft labels: the raters'
  top answer). For a score question this means the exact level.
- **distance**: how far the predicted probabilities are from the label (0 = identical, 1 = completely
  different).
- **log_loss**: cross-entropy against the label; lower is better.
- **score_error**: for score questions, the gap between predicted and true expected level.
- **auc**: for yes/no questions, how well the model ranks yes above no (0.5 = chance).

In [ ]:
compare(results, by="overall")

### Accuracy by domain

The number of test questions per domain is in `results["trained"]["by_category"][domain]["n"]`;
small domains give noisy numbers.

In [ ]:
compare(results, by="category").drop(columns="overall").T.sort_values("trained", ascending=False)

## What to expect

Training only the head on a frozen backbone is the cheapest setting and the least accurate: the
backbone was never adapted to these questions. Expect clear gains over random guessing but well
short of what fine-tuning the backbone gives. Score questions are the hardest, because the loss
treats levels as unrelated categories.

Things to try: more training rows (`N_TRAIN`), a different `num_layers` (default 0.7 of Qwen's 28
layers), and `tuning="lora"` / `"qlora"` once they are implemented.